# الدرس 13 - ذاكرة الوكيل


## الإعداد

يوضح هذا الدفتر كيفية بناء وكيل حجز سفر باستخدام **ذاكرة دائمة** باستخدام **إطار عمل وكيل مايكروسوفت** (MAF).

ستتعلم كيف تؤثر أنواع الذاكرة المختلفة للوكيل — العامل، قصيرة المدى، وطويلة المدى — على كيفية احتفاظ الوكيل بالمعلومات واستخدامها عبر المحادثات.

**المتطلبات المسبقة:**
- مشروع Microsoft Foundry مع نموذج دردشة مثبت (مثلاً `gpt-4.1-mini`).
- تسجيل الدخول باستخدام Azure CLI — شغل `az login` في الطرفية الخاصة بك.
- `AZURE_AI_PROJECT_ENDPOINT` — نقطة نهاية مشروع Microsoft Foundry الخاص بك.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج المثبت الخاص بك.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## أنواع ذاكرة الوكيل

يمكن لوكلاء الذكاء الاصطناعي الاستفادة من أنواع مختلفة من الذاكرة، كل منها يخدم غرضًا مميزًا:

### الذاكرة العاملة
سلسلة المحادثة نفسها — الرسائل المتبادلة في جلسة واحدة. يمكن للوكيل الرجوع إلى الرسائل السابقة في نفس السلسلة للحفاظ على التماسك. في MAF يتم إنشاء هذا عبر **`agent.create_session()`**، الذي يعيد `AgentSession`.

### الذاكرة القصيرة الأجل
المعلومات التي تدوم طوال مدة مهمة أو جلسة ولكن لا تُخزن بشكل دائم. على سبيل المثال، قد يجمع الوكيل حقائق خلال محادثة تخطيط متعددة الأدوار ويستخدمها لإنتاج جدول نهائي.

### الذاكرة الطويلة الأجل
التفضيلات والحقائق التي تدوم **عبر الجلسات**. لا ينبغي للمستخدم العائد أن يكرر قيود النظام الغذائي أو أسلوب السفر الخاص به. عادةً ما تدعم الذاكرة الطويلة الأجل بواسطة مخزن خارجي — قاعدة بيانات، ملف، أو فهرس متجه — ويتم تقديمها إلى الوكيل عبر أدوات.


## الذاكرة العاملة مع الجلسات

أبسط أشكال الذاكرة هي جلسة المحادثة. عند تمرير نفس الجلسة (التي تم إنشاؤها عبر `agent.create_session()`) إلى استدعاءات `agent.run()` المتتالية، يرى الوكيل التاريخ الكامل لتلك المحادثة ويمكنه استدعاء التفاصيل السابقة.

لننشئ وكيلاً للسفر ونوضح الذاكرة العاملة.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

استذكر العميل الميزانية بشكل صحيح لأن كلا الرسالتين تشتركان في نفس الجلسة. هذه هي **الذاكرة العاملة** — فهي موجودة فقط طوال مدة الجلسة.

### ماذا يحدث مع موضوع جديد؟

إذا أنشأنا جلسة **جديدة**، فلن يكون لدى العميل أي ذاكرة للمحادثة السابقة:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## نمط الذاكرة طويلة الأمد

لتذكر تفضيلات المستخدم **عبر الجلسات**، نحتاج إلى مخزن دائم يعيش خارج مسار المحادثة. يحصل الوكيل على هذه البيانات من خلال **أدوات** — وهي وظائف يمكنه استدعاؤها لحفظ واسترجاع المعلومات.

أدناه نقوم بتنفيذ مخزن تفضيلات بسيط في الذاكرة (في الإنتاج ستدعمه بقاعدة بيانات أو فهرس متجه) ونعرضه كأدوات يمكن للوكيل استخدامها.

### الهيكلية
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### السيناريو 1 — مستخدم يحجز رحلة ذكرى للمرة الأولى

سارة تزور للمرة الأولى. يجب على الوكيل تخزين تفضيلاتها عبر الأدوات واستخدامها لتوصية الفنادق.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### السيناريو 2 — عودة سارة بعد أسابيع

تبدأ سارة **سلسلة جديدة تمامًا** (تحاكي جلسة جديدة). ذاكرة العمل فارغة، لكن مخزن التفضيلات طويل الأمد لا يزال يحتوي على معلوماتها. يجب على الوكيل استرجاعها واستخدامها لتخصيص التوصيات.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ملخص

في هذا الدرس تعلمت ثلاثة أنواع من ذاكرة الوكيل وكيفية تنفيذها باستخدام إطار عمل Microsoft Agent:

| نوع الذاكرة | آلية MAF | مدة الحياة |
|---|---|---|
| **العاملة** | `agent.create_session()` | محادثة واحدة |
| **قصيرة الأمد** | السياق المتراكم داخل سلسلة المحادثة | مهمة / جلسة واحدة |
| **طويلة الأمد** | تخزين خارجي يتم الوصول إليه عبر دوال `@tool` | عبر الجلسات |

### النقاط الرئيسية المستفادة
1. **`agent.create_session()`** يوفر ذاكرة عاملة — يرى الوكيل كامل سجل المحادثة ضمن الجلسة.
2. **الجلسات الجديدة تفقد السياق** — بدون ذاكرة طويلة الأمد لا يستطيع الوكيل تذكر المحادثات السابقة.
3. **دوال `@tool`** تسد الفجوة — تسمح للوكيل بحفظ واسترجاع المعلومات من مخزن دائم.
4. **التحسين الشخصي يتحسن مع الوقت** — كلما تم تخزين المزيد من التفضيلات، تحسنت توصيات الوكيل.

### التطبيقات العملية
- **خدمة العملاء**: تذكر تاريخ العميل وتفضيلاته
- **المساعدون الشخصيون**: الحفاظ على السياق عبر أيام أو أسابيع
- **الرعاية الصحية**: تتبع معلومات وتفضيلات المريض
- **التجارة الإلكترونية**: تسوق مخصص بناءً على التاريخ

### الخطوات التالية
- استبدال القاموس في الذاكرة بقاعدة بيانات أو مخزن متجهات (مثلاً Azure AI Search)
- إضافة انتهاء صلاحية الذاكرة للمعلومات الحساسة للوقت
- بناء أنظمة متعددة الوكلاء بذاكرة مشتركة
- استكشاف دفتر Cognee لذاكرة مدعومة برسم المعرفة


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
